# Resolviendo N-Queens con GA usando DEAP

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Implementar** un Algoritmo Genético con la librería DEAP (`creator`, `toolbox`, `tools`).
2. **Resolver** el problema de las N-Reinas mediante una representación por permutación.
3. **Configurar** los operadores de selección, cruza y mutación en DEAP.
4. **Ejecutar** `algorithms.eaSimple` y recolectar estadísticas de la evolución con `tools.Statistics`.
5. **Analizar** la curva de convergencia hacia una solución sin conflictos.

> 🧪 **Laboratorio:** contraparte práctica del módulo de Computación Evolutiva.

> 💡 **Prerrequisitos:** [03_ComputacionEvolutiva](03_ComputacionEvolutiva.ipynb).

## Distributed Evolutionary Algorithms in Python (DEAP)

**DEAP** es un framework de Python para prototipar Algoritmos Evolutivos rápidamente: en lugar de programar cada operador desde cero, se **ensamblan** componentes ya implementados. Nos apoyaremos en tres módulos:

* **`creator`**: crea dinámicamente nuevos tipos (la clase de *fitness* y la clase de *individuo*).
* **`base`**: aporta el **`Toolbox`**, un contenedor donde registramos funciones y operadores con un nombre propio.
* **`tools`**: provee operadores evolutivos listos para usar (selección, cruza, mutación) y utilidades de estadística.

Notas:
* Framework para prototipado y testeo de ideas rápidas. Documentación: https://deap.readthedocs.io/en/master/index.html
* Para producción de alto rendimiento, los algoritmos evolutivos suelen implementarse en C/C++ (mejor manejo de memoria, *speed-up* y del espacio de búsqueda).
* DEAP ya trae algoritmos y elementos clásicos de los EAs implementados.

## Representación

<center><img src="images/nqueens.png" alt="" width="20%" align="center"/></center>

Vector de enteros.
* Gen: Columnas.
* Alelos: Filas.
* Evaluación: # Checks

Ventajas:
* No habrán checks en columnas.

> $[x_1, x_2, x_3, \dots, x_n]$ con $n =$ ```board_size```.

> $x_i:$ Fila que ocupa una reina en la columna $i$, $x_i \in \{1,2,\dots,n\}$

* Gen: Columnas.
* Alelos: Filas.
* Evaluación: # Checks

Ventajas:
* No habrán checks en columnas.

## Importar Bibliotecas

In [ ]:
#importamos bibliotecas
import numpy as np
import random as rd
from deap import creator as cr, base as bs, tools as tl

# creator: permite definir nuevos tipos de individuos y fitness  
# base: contiene el Toolbox, donde se registran funciones personalizadas  
# tools: provee operadores evolutivos listos para usar (selección, mutación, cruzamiento)

## Semilla Aleatoria: Reproductibilidad

In [ ]:
#semilla aleatoria para reproducir resultados
rd.seed(0)
print(rd.randint(1,10))

## Creando las clases Fitness e Individuo

Con **`creator.create(nombre, base, **atributos)`** definimos, en tiempo de ejecución, nuevos tipos:

* La clase **`Fitness`** hereda de `base.Fitness`. Su atributo `weights` es una **tupla** cuyo signo indica el sentido de la optimización: `(-1.0,)` = **minimizar** una función objetivo, `(1.0,)` = maximizar. Con varias entradas se modela un problema **multiobjetivo**.
* La clase **`Individuo`** hereda de `list` (el cromosoma es una lista de genes) y lleva asociado un objeto `Fitness`.

Para N-Reinas minimizamos el número de reinas en jaque, por eso usamos `weights=(-1.0,)`.

In [ ]:
#Creamos Clase de Función Fitness: Hereda de base.Fitness
#Fitness tiene funciones de evaluación en tuplas:
    #(-1.0,) indica 1 función de evaluación en Min.
    #(1.0, -1.0) indica multiobjetivo, 1 función de evaluación en Max, y 2da funcion de evaluación en Min.
cr.create("Fitness", bs.Fitness, weights=(-1.0,)) 

#Creamos Clase de Individuo: Hereda de lista y usara la clase Fitness creada para ser evaluado.
cr.create("Individuo", list, fitness=cr.Fitness) 

## Seteando el Tablero y funciones base

El **`Toolbox`** es el "kit de herramientas" del algoritmo: con **`toolbox.register(alias, funcion, *args)`** guardamos una función bajo un alias y con sus argumentos ya fijados. Luego la invocamos como `toolbox.alias(...)`.

Aquí registramos cómo se **construye un individuo**:

* `fill_int`: genera el valor (fila) de un gen con `random.randint(0, board_size-1)`.
* `fill_ind`: usa `tools.initRepeat` para crear un `Individuo` repitiendo `fill_int` una vez por columna (`n = board_size`).

In [ ]:
# tamaño tablero
board_size = 6 

# creando funciones para para modelo evolutivo.
toolbox = bs.Toolbox() 

# se define fill_int: función para inicializar un gen, usa randint entre 0 y board_size-1
toolbox.register("fill_int", rd.randint, 0, board_size-1) 

#se define fill_ind: función para inicializar un individuo
#repite (initRepeat) por cada gen (cant genes = board_size) de Individuo usando la función fill_int 
toolbox.register("fill_ind", tl.initRepeat, cr.Individuo, toolbox.fill_int, n=board_size) 

## Creando nuestro primer individuo 

In [ ]:
#creamos un individuo usando función fill_ind del toolbox
ind1 = toolbox.fill_ind()

print(type(ind1))
#imprimimos el individuo
print(ind1)

#vemos si tiene un fitness válido (no hemos definido fitness)
print(ind1.fitness.valid)

## Función de Evaluación

Lo más complejo que hay que programar es la Función Fitness. No hay formas pre-implementadas. 

### Función de Checks en Diagonales

In [ ]:
# get_diag: dada una reina (gen=fila, pos=columna) devuelve las casillas de sus dos diagonales dentro del tablero
def get_diag(gen, pos, bsize):
    bsize = bsize -1
    L=[]
    #1/4 diag
    row = gen-1
    col = pos+1
    while (0 <= row <= bsize) and (0 <= col <= bsize):
        L.append((col,row))
        row-=1
        col+=1
    
    #2/4 diag
    row = gen+1
    col = pos+1
    while (0 <= row <= bsize) and (0 <= col <= bsize):
        L.append((col,row))
        row+=1
        col+=1
### Eliminadas por redundancia
#    #3/4 diag
#    row = gen+1
#    col = pos-1
#    while (0 <= row <= bsize) and (0 <= col <= bsize):
#        L.append((col,row))
#        row+=1
#        col-=1
#    
#    #4/4 diag
#    row = gen-1
#    col = pos-1
#    while (0 <= row <= bsize) and (0 <= col <= bsize):
#        L.append((col,row))
#        row-=1
#        col-=1        
    return L

### Función Fitness

In [ ]:
# evaluate: función de fitness. Cuenta el nº de reinas en jaque por filas y por diagonales (se minimiza -> 0 = solución)
def evaluate(individual):
    check = 0
    index = 1
    #contamos checks en filas
    for i in individual:
        #print(individual[index:])
        check += individual[index:].count(i)
        #print(check)
        index+=1
    #contamos checks en diagonales
    for i in range(len(individual)):
        diags = get_diag(individual[i], i, board_size)
        for col, row in diags:
            if individual[col] == row:
                check +=1       
        
    
    return (check,) #debe retornar una tupla por como trabaja la clase Fitness.

### Evaluamos individuo 

In [ ]:
# Asignamos el fitness al individuo ind1 evaluándolo con la función evaluate
ind1.fitness.values = evaluate(ind1)

In [ ]:
# Comprobamos que el fitness quedó registrado (valid=True) y lo mostramos
print(ind1.fitness.valid)
print(ind1.fitness)

## Población

In [ ]:
# Creamos funcion fill_pop que usando initRepeat crea una lista que se llena utilizando find_ind por cada elemento. 
# Lista de Individuos = Población
toolbox.register("fill_pop", tl.initRepeat, list, toolbox.fill_ind)

In [ ]:
# Población de 20 Individuos
pobla = toolbox.fill_pop(20)
print(pobla)

In [ ]:
# funcion para evaluar todos los individuos de una población y la registramos en la toolbox
def eval_pop(pop):
    for i in pop:
        i.fitness.values = evaluate(i)

toolbox.register("evaluar", eval_pop)
toolbox.evaluar(pobla)

In [ ]:
# funcion para imprimir población, con su fitness, y la registramos en la toolbox
def print_pobla(pop):
    ind = 1
    for i in pop:
        print(ind, i, i.fitness)
        ind+=1

toolbox.register("imprimir_pop", print_pobla)
toolbox.imprimir_pop(pobla)

## Operadores

La clase tools contiene pre-establecido una gran cantidad de operadores genéticos para poder ser utilizados.
* https://deap.readthedocs.io/en/master/api/tools.html#module-deap.tools

### Selección de Padres

In [ ]:
#registramos selección como un Torneo de 4 participantes aleatorios para obtener k=4 padres. 1 Torneo por padre.
toolbox.register("seleccion", tl.selTournament, k=4, tournsize=4)
padres=toolbox.seleccion(pobla)
# La seleccion es in place, por lo tanto si modificamos padres, modificaremos pobla.
# Clonamos padres para no modificar población original. 
padres_clon=list(map(toolbox.clone,padres)) 
toolbox.imprimir_pop(padres_clon)
print("-----")
toolbox.imprimir_pop(pobla)

### Recombinación

In [ ]:
# Registramos la función rex para recombinación en dos puntos
toolbox.register("rex", tl.cxTwoPoint)
# Proceso para ir cruzando de 2 en dos padres de forma correlativa. 
for i in range(0,len(padres_clon),2):
    toolbox.rex(padres_clon[i],padres_clon[i+1])
    del padres_clon[i].fitness.values, padres_clon[i+1].fitness.values #borramos el valor fitness heredado desde el padre
    
toolbox.imprimir_pop(padres_clon)
#print("-----")
#toolbox.imprimir_pop(pobla)

### Mutación

In [ ]:
# Registramos la función mux como Mutación Entera uniforme
toolbox.register("mux", tl.mutUniformInt, low=0, up=5)
for i in range(len(padres_clon)):
    toolbox.mux(padres_clon[i], indpb=0.3) # cada gen del individuo tiene un 30% de probabilidad de ser mutado.
    del padres_clon[i].fitness.values #borramos el fitnes heredado del padre
toolbox.imprimir_pop(padres_clon)
#print("-----")
#toolbox.imprimir_pop(pobla)

### Supervivencia

In [ ]:
# imprimimos situación actual
toolbox.imprimir_pop(padres)
print("----")
toolbox.evaluar(padres_clon)
toolbox.imprimir_pop(padres_clon)

In [ ]:
# Mostramos la población antes y después de reemplazar los padres por los hijos (efecto in place sobre 'pobla')
toolbox.imprimir_pop(pobla)
print("----- Cambiando padres por hijos -----")
#hacemos el cambio desde padres_clon a padres de tal forma de modificar pobla.
for i in range(len(padres)):
    padres[i].fitness = padres_clon[i].fitness
    for j in range(len(padres[i])):
        padres[i][j] = padres_clon[i][j]
        
toolbox.imprimir_pop(pobla)

## Ejercicio: integre las partes y construya el Algoritmo Evolutivo

Con los bloques anteriores (representación, evaluación, selección, cruza, mutación y supervivencia) ya tiene todo para armar el ciclo completo. Puede orquestarlo manualmente o usar el algoritmo pre-implementado de DEAP.

**`algorithms.eaSimple(pop, toolbox, cxpb, mutpb, ngen, stats, halloffame)`** ejecuta el GA generacional clásico. Requiere que en el `toolbox` estén registrados con los alias estándar: `evaluate`, `select`, `mate` y `mutate`. Sus parámetros clave son la probabilidad de cruza `cxpb`, la de mutación `mutpb` y el número de generaciones `ngen`.

**`tools.Statistics`** (junto con `tools.Logbook`) recolecta métricas de la población en cada generación (por ejemplo `min`, `avg`, `max` del fitness con `numpy`), lo que permite luego graficar la **curva de convergencia**.

**Tareas:**

* Decida los **criterios de detención** (p. ej. `ngen` o alcanzar fitness $0$).
* Decida las **probabilidades** de aplicación de los operadores (`cxpb`, `mutpb`).
* Registre y muestre estadísticas de la población con `tools.Statistics`:
  https://deap.readthedocs.io/en/master/tutorials/basic/part3.html
* Grafique el fitness (mejor y promedio) vs. la generación e incluya `xlabel`, `ylabel`, `title` y `legend` en la figura.

In [ ]:
# Coloque su código aquí.

## 📌 Resumen

| Componente DEAP | Rol |
|-----------------|-----|
| `creator.create` | Define en runtime las clases `Fitness` (con `weights`) e `Individuo` (lista de genes). |
| `base.Toolbox` + `register` | Registra funciones/operadores bajo un alias con argumentos fijados. |
| `tools.initRepeat` | Construye individuos y poblaciones repitiendo una función generadora. |
| Función `evaluate` | Fitness propio del problema (N-Reinas: nº de reinas en jaque, a minimizar). |
| `tools.selTournament` | Selección de padres por torneo. |
| `tools.cxTwoPoint` | Recombinación (cruza) en dos puntos. |
| `tools.mutUniformInt` | Mutación entera uniforme por gen (`indpb`). |
| `algorithms.eaSimple` | Ciclo GA generacional completo (`cxpb`, `mutpb`, `ngen`). |
| `tools.Statistics` / `Logbook` | Recolección de métricas para la curva de convergencia. |

## 🔗 Próximos pasos

- Complete el ejercicio final ensamblando el GA con `algorithms.eaSimple` y grafique la curva de convergencia.
- Repase la teoría en [03_ComputacionEvolutiva.ipynb](03_ComputacionEvolutiva.ipynb).

## 📚 Referencias

- **[DEAP]** Documentación oficial: https://deap.readthedocs.io/en/master/
- **[DEAP · Operadores]** Módulo `tools`: https://deap.readthedocs.io/en/master/api/tools.html
- **[DEAP · Estadísticas]** Tutorial *Computing Statistics*: https://deap.readthedocs.io/en/master/tutorials/basic/part3.html
- **[Fortin et al., 2012]** *DEAP: Evolutionary Algorithms Made Easy*. Journal of Machine Learning Research, 13, 2171–2175.
- **[Eiben & Smith, 2015]** *Introduction to Evolutionary Computing* (2ª ed.). Springer.